In [ ]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="qwen3.5-27b",
    )

    answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are a Villager. 
Goal: Eliminate all Mafia members.
Format your output EXACTLY as shown below, without any additional text or explanation. :
<thinking>
[Your reasoning]
</thinking>
[Player X]""",
    user_message="""
Phase: Day 1 - Voting
Chat History:
Player1: Player2 is acting highly suspicious.
Player2: I am just a normal Villager!
Observation: It is your turn to vote.""")
    print("RESPONSE:", answer)
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
It is Day 1 with very little information. Player1 is making a direct accusation against Player2 without providing any evidence or reasoning. This aggressive behavior early in the game is often a tactic used by Mafia members to create chaos and frame an innocent Villager. Player2 simply denied the accusation, which is a standard response. To protect the village from potential manipulation, I will vote for the accuser.
</thinking>
[Player 1]


In [6]:
answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are a Mafia. 
Goal: Eliminate all Villagers.
Format your output EXACTLY as shown below, without any additional text or explanation. :
<thinking>
[Your reasoning]
</thinking>
[Player X]""",
    user_message="""
Phase: Day 1 - Night
Observation: It is your turn to eliminate a Villager. Your Mafia teammate is [Player 3]. Valid targets are [Player 1, Player 2, Player 4, Player 5].""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
It is Night 1. I am a Mafia member and my goal is to eliminate Villagers. My teammate is Player 3, so I must avoid targeting them. The valid targets provided are Player 1, Player 2, Player 4, and Player 5. Since I have no specific information about the roles of these players yet, I will randomly select one from the valid list to eliminate. I choose Player 4.
</thinking>
[Player 4]
